# Caso 23F: segmentación inicial de conversaciones en frases

Objetivo de este notebook:

1. Leer el CSV scrapeado de documentos del 23F.
2. Explorar rápidamente su estructura.
3. Limpiar de forma básica el texto OCR.
4. Dividir cada `transcript` en frases candidatas.
5. Guardar una tabla de frases que luego servirá para etiquetar: `A favor`, `En contra` o `Neutral`.

> Nota: aquí todavía no entrenamos ningún modelo. Primero necesitamos construir una unidad de análisis razonable: las frases.

In [ ]:
# Librerías básicas
import pandas as pd
import re
from pathlib import Path

pd.set_option("display.max_colwidth", 250)
pd.set_option("display.max_rows", 100)

## 1. Cargar el CSV

In [ ]:
# Ruta del dataset original
DATA_PATH = Path("/mnt/data/23f_scrappedDF.csv")

df = pd.read_csv(DATA_PATH)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head(3)

## 2. Revisión rápida del dataset

In [ ]:
# Columnas disponibles
df.columns.tolist()

In [ ]:
# Valores nulos por columna
df.isna().sum()

In [ ]:
# Longitud aproximada de cada transcripción
df["transcript_len"] = df["transcript"].astype(str).str.len()
df["transcript_len"].describe()

In [ ]:
# Ver algunos documentos cortos/largos
df[["title", "filename", "pages", "transcript_len"]].sort_values("transcript_len").head(10)

In [ ]:
df[["title", "filename", "pages", "transcript_len"]].sort_values("transcript_len", ascending=False).head(10)

## 3. Limpieza básica del texto OCR

Esta limpieza es conservadora: no queremos borrar información útil antes del etiquetado. De momento corregimos espacios, saltos de línea y algunos patrones típicos de OCR.

In [ ]:
def clean_ocr_text(text: str) -> str:
    """Limpieza básica para texto OCR de documentos históricos."""
    if pd.isna(text):
        return ""
    text = str(text)

    # Normalizar saltos de línea y espacios
    text = text.replace("\r", "\n")
    text = re.sub(r"[\t ]+", " ", text)

    # Unir palabras cortadas por guion al final de línea: "comunica-\nción" -> "comunicación"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Sustituir múltiples saltos de línea por punto y espacio si parecen separar bloques
    # Esto ayuda a que el segmentador encuentre frases, pero mantiene separación semántica.
    text = re.sub(r"\n{2,}", ". ", text)

    # Saltos de línea simples a espacio
    text = re.sub(r"\n", " ", text)

    # Espacios múltiples
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_transcript"] = df["transcript"].apply(clean_ocr_text)
df[["title", "clean_transcript"]].head(2)

## 4. División en frases

Para empezar usamos una función basada en expresiones regulares. No es perfecta, pero es transparente y fácil de ajustar.

Más adelante podemos mejorarla con `spaCy` o `NLTK`, pero para una primera versión académica esta aproximación suele ser suficiente.

In [ ]:
# Abreviaturas frecuentes que pueden romper mal las frases
ABBREVIATIONS = [
    "D.", "Dª.", "Sr.", "Sra.", "Sres.", "Excmo.", "Ilmo.",
    "Gral.", "Tcol.", "TCOL.", "Cap.", "Cte.", "Cor.", "Dtor.",
    "Art.", "Núm.", "nº.", "etc."
]

def protect_abbreviations(text: str) -> str:
    """Protege puntos de abreviaturas para evitar cortes incorrectos."""
    protected = text
    for abbr in ABBREVIATIONS:
        safe = abbr.replace(".", "<DOT>")
        protected = protected.replace(abbr, safe)
    return protected


def restore_abbreviations(text: str) -> str:
    return text.replace("<DOT>", ".")


def split_into_sentences(text: str, min_chars: int = 30, max_chars: int = 600) -> list[str]:
    """Divide texto en frases candidatas.

    Parámetros:
    - min_chars: elimina fragmentos demasiado cortos.
    - max_chars: si una frase es demasiado larga, se intenta dividir por ; o :.
    """
    text = clean_ocr_text(text)
    text = protect_abbreviations(text)

    # Cortar tras punto, interrogación o exclamación cuando después viene espacio y mayúscula/número/guion
    parts = re.split(r"(?<=[\.\?\!])\s+(?=[A-ZÁÉÍÓÚÜÑ0-9\-])", text)
    parts = [restore_abbreviations(p).strip() for p in parts]

    final_parts = []
    for p in parts:
        if len(p) > max_chars:
            subparts = re.split(r"(?<=[;:])\s+", p)
            final_parts.extend(subparts)
        else:
            final_parts.append(p)

    # Filtros básicos
    final_parts = [p.strip(" -•\t") for p in final_parts]
    final_parts = [p for p in final_parts if len(p) >= min_chars]

    return final_parts


# Prueba rápida con el primer documento
example_sentences = split_into_sentences(df.loc[0, "transcript"])
len(example_sentences), example_sentences[:10]

## 5. Crear tabla de frases

Cada fila será una frase candidata, manteniendo metadatos del documento original. Esta será la tabla que después podremos etiquetar manualmente.

In [ ]:
sentence_rows = []

for doc_id, row in df.iterrows():
    sentences = split_into_sentences(row["transcript"])
    for sent_id, sentence in enumerate(sentences, start=1):
        sentence_rows.append({
            "doc_id": doc_id,
            "sentence_id": sent_id,
            "title": row.get("title", ""),
            "filename": row.get("filename", ""),
            "pages": row.get("pages", ""),
            "sentence": sentence,
            "sentence_len": len(sentence),
            # Columna vacía para etiquetado posterior
            "label": ""
        })

sentences_df = pd.DataFrame(sentence_rows)

print(f"Total de frases candidatas: {len(sentences_df)}")
sentences_df.head(20)

In [ ]:
# Distribución de longitud de las frases
sentences_df["sentence_len"].describe()

In [ ]:
# Muestra aleatoria para revisar calidad de segmentación
sentences_df.sample(20, random_state=42)[["doc_id", "sentence_id", "sentence_len", "sentence", "label"]]

## 6. Filtros opcionales de calidad

Algunas frases candidatas pueden ser ruido: códigos, encabezados, páginas, fechas o texto administrativo. Aquí añadimos una función simple para marcar posibles frases poco útiles. No las borramos automáticamente todavía.

In [ ]:
def flag_possible_noise(sentence: str) -> bool:
    """Marca posibles fragmentos ruidosos del OCR o encabezados administrativos."""
    s = sentence.strip()

    # Demasiados caracteres no alfabéticos
    letters = sum(ch.isalpha() for ch in s)
    ratio_letters = letters / max(len(s), 1)
    if ratio_letters < 0.45:
        return True

    # Patrones administrativos frecuentes
    noise_patterns = [
        r"^C/", r"^ANEXO", r"^NOTA INFORMATIVA$", r"^ASUNTO:",
        r"^P[ÁA]GINA", r"^DOCUMENTO", r"^FECHA:", r"^REF",
    ]
    if any(re.search(p, s, flags=re.IGNORECASE) for p in noise_patterns):
        return True

    return False


sentences_df["possible_noise"] = sentences_df["sentence"].apply(flag_possible_noise)
sentences_df["possible_noise"].value_counts()

In [ ]:
# Revisar ejemplos marcados como posible ruido
sentences_df[sentences_df["possible_noise"]].sample(
    min(20, sentences_df["possible_noise"].sum()),
    random_state=7
)[["doc_id", "sentence_id", "sentence_len", "sentence", "possible_noise"]]

## 7. Exportar frases para etiquetado

Guardamos dos versiones:

- `23f_frases_candidatas.csv`: incluye todo.
- `23f_frases_para_etiquetar.csv`: excluye lo marcado como posible ruido.

In [ ]:
OUTPUT_ALL = Path("/mnt/data/23f_frases_candidatas.csv")
OUTPUT_LABEL = Path("/mnt/data/23f_frases_para_etiquetar.csv")

sentences_df.to_csv(OUTPUT_ALL, index=False)
sentences_df[~sentences_df["possible_noise"]].to_csv(OUTPUT_LABEL, index=False)

print("Archivos guardados:")
print(OUTPUT_ALL)
print(OUTPUT_LABEL)

## Siguiente paso

Revisar una muestra de frases y definir criterios de etiquetado:

- `A favor`: apoyo explícito o justificación del golpe / acciones golpistas.
- `En contra`: rechazo explícito, oposición, defensa del orden constitucional.
- `Neutral`: descripción de hechos, información administrativa, declaraciones ambiguas o sin postura clara.

Después conviene etiquetar manualmente una muestra inicial y entrenar un primer modelo base.